# Milestone 5 — The ball: cleaning, filling gaps, and possession

**Problem after Milestone 4:** we know every player and his team, but the ball is the hardest object in the clip. It is tiny, fast, often hidden behind legs, and the detector sometimes boxes a white boot or the penalty spot instead. Without a reliable ball there is no possession.

**Plan**
1. Concepts
2. Load the table from Milestone 4
3. Look at the raw ball detections
4. Remove false balls (outliers)
5. Fill short gaps (interpolation)
6. Who has the ball? (possession per frame)
7. Smooth it, compute possession %, passes and turnovers
8. Experiments
9. Render the video and save the tables
10. Check yourself

**Input:** `outputs/m4_tracks_teams.csv` and the video. No GPU needed.

## 1. Concepts

**Ball table.** One row per frame: the ball's centre `(cx, cy)` or NaN when it was not detected. Milestone 3 already kept only the most confident ball per frame.

**False balls (outliers).** A real ball moves along a smooth path. A false detection is a single jump away from that path. For each detection we compute the **median** position of its neighbours (a window of 7 detections) and flag it if it is more than `max_dev` pixels away.
> Why the median? On a straight line, the median of 7 consecutive points is exactly the middle point, so real fast movement is not flagged. One wrong point cannot move the median, but it would move the mean. Same robustness idea as the median shirt colour in Milestone 4.

**Filling gaps (interpolation).** If the ball is at A in frame 100 and at B in frame 104, it was probably on the line between A and B in frames 101 to 103. We fill **only short gaps** (`max_gap`, default 25 frames = 1 s) with a known ball on **both** sides. A long gap usually means the ball left the picture or flew high: drawing a line there would invent data.
> Tabular analogy: filling missing values in a time series with linear interpolation, but refusing to fill long holes.

**Who has the ball?** In each frame, take every player's **feet** (bottom-centre of his box) and measure the distance to the ball. Divide by the player's **box height**, so the rule works whether the camera is zoomed in or out. The closest player with `distance / height ≤ max_ratio` (default 0.5, half a body height) has the ball.

**Smoothing.** Two rules:
- a holder must keep the ball for `min_frames` frames in a row (a player running past the ball for one frame is not a touch);
- during a pass, nobody is close to the ball, so the **team that touched it last stays in control** until the other team touches it.

**Passes and turnovers.** Consecutive touches: same team, different player = **pass**; other team = **turnover**.

**Honest limits:** everything is in **pixels**, from one camera. A high ball can look "at the feet" of a player it is flying over; a player far from the camera looks small. Milestone 6 (pitch coordinates in metres) improves this.

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
from football_tracking.ball import (ball_table, flag_outliers, interpolate_ball, gap_lengths, assign_possession,
    smooth_possession, possession_stats, touch_events, running_possession, PossessionAnnotator)
from football_tracking.teams import TeamAnnotator, sample_rows, collect_features
from football_tracking.video_io import video_info, iter_frames, VideoWriter

CLIP_NAME = "08fd33_0.mp4"
VIDEO = ROOT / "data" / "raw" / CLIP_NAME
TABLE = ROOT / "outputs" / "m4_tracks_teams.csv"
OUT = ROOT / "outputs"
assert VIDEO.exists(), VIDEO
assert TABLE.exists(), f"{TABLE} missing: run section 10 of notebook 04 first"

CLS = "voted_class_name"
TEAM_NAMES = {0: "Green", 1: "White"}        # same names as in notebook 04
df = pd.read_csv(TABLE)
info = video_info(VIDEO)
N = info["frame_count"]
print(info)
print(df[CLS].value_counts())

In [ ]:
# team colours for drawing: the median shirt colour of each team (same method as notebook 04)
rows = sample_rows(df[df.team >= 0], class_col=CLS, classes=("player",), per_track=5)
f_, _ = collect_features(VIDEO, rows)
f_ = f_.merge(df[["tracker_id", "team"]].drop_duplicates("tracker_id"), on="tracker_id")
TEAM_COLOURS = {int(t): tuple(int(v) for v in g[["B", "G", "R"]].median()) for t, g in f_.groupby("team")}
print("team colours (BGR):", TEAM_COLOURS)

def read_frame(i):
    cap = cv2.VideoCapture(str(VIDEO)); cap.set(cv2.CAP_PROP_POS_FRAMES, i)
    ok, fr = cap.read(); cap.release(); assert ok; return fr

## 2. The raw ball detections

In [ ]:
ball = ball_table(df, N, class_col=CLS)
print(f"ball detected in {ball.detected.mean():.0%} of frames ({int(ball.detected.sum())} of {N})")
gl = gap_lengths(~ball.detected)
runs = gl[gl > 0].groupby((gl == 0).cumsum()).first()
print("gaps without a ball:", len(runs), "| longest:", int(runs.max()) if len(runs) else 0, "frames")

fig, axs = plt.subplots(2, 1, figsize=(12, 4.5), sharex=True)
for ax, col in zip(axs, ["cx", "cy"]):
    ax.scatter(ball.index, ball[col], s=6, c="#1d7447"); ax.set_ylabel(f"ball {col} (px)"); ax.grid(alpha=.25)
axs[1].set_xlabel("frame"); axs[0].set_title("Raw ball detections (gaps = not detected; isolated jumps = suspicious)")
plt.tight_layout(); plt.show()

**How to read it:** a real ball draws continuous curves. Single dots far from the curve are probably false balls. Empty stretches are frames where the detector did not find the ball.

## 3. Remove false balls

In [ ]:
MAX_DEV = 40          # pixels away from the rolling median of nearby detections
out = flag_outliers(ball, window=7, max_dev=MAX_DEV)
print(int(out.sum()), "detections flagged as false balls:", list(ball.index[out])[:20])

fig, axs = plt.subplots(2, 1, figsize=(12, 4.5), sharex=True)
for ax, col in zip(axs, ["cx", "cy"]):
    ax.scatter(ball.index[~out], ball[col][~out], s=6, c="#1d7447", label="kept")
    ax.scatter(ball.index[out], ball[col][out], s=40, c="#b3372a", marker="x", label="flagged")
    ax.set_ylabel(col); ax.grid(alpha=.25)
axs[0].legend(loc="upper right"); axs[1].set_xlabel("frame"); plt.tight_layout(); plt.show()

In [ ]:
# what were the flagged "balls"? (a crop around each one)
flagged = ball[out].head(8)
if len(flagged):
    fig, axs = plt.subplots(1, len(flagged), figsize=(1.8 * len(flagged), 2.2))
    for ax, (f, r) in zip(np.atleast_1d(axs), flagged.iterrows()):
        fr = read_frame(int(f)); x, y = int(r.cx), int(r.cy)
        crop = fr[max(y - 40, 0):y + 40, max(x - 40, 0):x + 40]
        ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)); ax.set_title(f"frame {f}", fontsize=8); ax.axis("off")
    plt.suptitle("flagged detections (the ball should NOT be at the centre)"); plt.tight_layout(); plt.show()

**Check:** if a crop clearly shows the real ball at its centre, `MAX_DEV` is too strict for this clip (fast shots); raise it. If false balls stay in the plot, lower it.

## 4. Fill short gaps

In [ ]:
MAX_GAP = 25          # frames (1 s at 25 fps)
bi = interpolate_ball(ball, out, max_gap=MAX_GAP)
print(bi.source.value_counts().to_dict())
print(f"ball position known in {bi.x.notna().mean():.0%} of frames (was {ball.detected.mean():.0%})")

colours = {"detected": "#1d7447", "interpolated": "#e08a00", "outlier": "#b3372a"}
fig, axs = plt.subplots(2, 1, figsize=(12, 4.5), sharex=True)
for ax, col in zip(axs, ["x", "y"]):
    for src, c in colours.items():
        m = bi.source == src
        ax.scatter(bi.index[m], bi[col][m] if src != "outlier" else bi["c" + col][m], s=8 if src != "outlier" else 30,
                   c=c, marker="x" if src == "outlier" else "o", label=src)
    ax.set_ylabel(col); ax.grid(alpha=.25)
axs[0].legend(loc="upper right", ncol=3); axs[1].set_xlabel("frame")
axs[0].set_title("Cleaned ball path: green = detected, orange = filled, red x = removed"); plt.tight_layout(); plt.show()

## 5. Who has the ball?

First look at the numbers behind the rule: for every frame with a ball, the **closest player's** distance divided by his height. Frames where someone has the ball form a peak at small values. Choose `MAX_RATIO` in the valley after that peak.

In [ ]:
raw = assign_possession(df, bi, class_col=CLS, max_ratio=99)      # 99 = no limit, just to see the distances
plt.figure(figsize=(9, 3))
plt.hist(raw.ratio.dropna().clip(upper=3), bins=60, color="#1d7447")
plt.axvline(0.7, color="#b3372a", ls="--", label="chosen MAX_RATIO = 0.7")
plt.xlabel("closest player: distance to ball / player height"); plt.ylabel("frames"); plt.legend(); plt.grid(alpha=.25)
plt.title("How close is the nearest player to the ball?"); plt.show()

In [ ]:
MAX_RATIO = 0.7                               # 0.5 missed touches with the ball just ahead of the feet on the sample clip
MIN_FRAMES = 3
poss = smooth_possession(assign_possession(df, bi, class_col=CLS, max_ratio=MAX_RATIO), min_frames=MIN_FRAMES)
print(f"someone has the ball in {(poss.holder_id_s >= 0).mean():.0%} of frames")
stats = possession_stats(poss).rename(index=TEAM_NAMES)
stats

In [ ]:
# possession over time: who is in control in each frame
cmap = {t: tuple(c / 255 for c in TEAM_COLOURS[t][::-1]) for t in TEAM_COLOURS}
fig, ax = plt.subplots(figsize=(12, 1.6))
for t, c in cmap.items():
    m = poss.team_in_control == t
    ax.bar(poss.index[m], 1, width=1.0, color=c, edgecolor="none", label=TEAM_NAMES.get(t, t))
held = poss.holder_id_s >= 0
ax.scatter(poss.index[held], np.full(held.sum(), 1.08), s=2, c="black", label="a player touches the ball")
ax.set_yticks([]); ax.set_xlim(0, N); ax.set_xlabel("frame"); ax.legend(loc="upper right", ncol=3, fontsize=8)
ax.set_title("Team in control per frame (gaps before the first touch are blank)"); plt.tight_layout(); plt.show()

In [ ]:
events = touch_events(poss)
events["team"] = events.team.map(TEAM_NAMES)
print(events.event.value_counts().to_dict())
print(events.groupby(["team", "event"]).size().unstack(fill_value=0))
events.head(20)

**Check against the video:** pick 3 events from this table, jump to their `start` frame (seconds = frame / 25) and confirm. Note whether passes are real passes and turnovers real losses of the ball.

## 6. Experiments

Change one setting at a time. The headline number (possession %) should be **stable** across reasonable settings; if it swings a lot, the result is not trustworthy yet.

In [ ]:
def run(max_dev=MAX_DEV, max_gap=MAX_GAP, max_ratio=MAX_RATIO, min_frames=MIN_FRAMES):
    b = interpolate_ball(ball, flag_outliers(ball, max_dev=max_dev), max_gap=max_gap)
    p = smooth_possession(assign_possession(df, b, class_col=CLS, max_ratio=max_ratio), min_frames=min_frames)
    s = possession_stats(p)["possession_%"]
    e = touch_events(p).event.value_counts()
    return {"ball_known_%": round(b.x.notna().mean() * 100), "holder_%": round((p.holder_id_s >= 0).mean() * 100),
            **{f"poss_{TEAM_NAMES.get(t, t)}": v for t, v in s.items()},
            "passes": int(e.get("pass", 0)), "turnovers": int(e.get("turnover", 0))}

settings = {"default": {}, "max_ratio 0.3": {"max_ratio": 0.3}, "max_ratio 0.8": {"max_ratio": 0.8},
            "min_frames 1": {"min_frames": 1}, "min_frames 6": {"min_frames": 6},
            "max_gap 10": {"max_gap": 10}, "max_gap 50": {"max_gap": 50},
            "max_dev 20": {"max_dev": 20}, "max_dev 80": {"max_dev": 80}}
pd.DataFrame({k: run(**v) for k, v in settings.items()}).T

**What to look for**
- `min_frames 1` usually creates many fake turnovers (a defender's foot is near the ball for one frame).
- A large `max_ratio` gives the ball to players who are only near it; a small one misses real touches.
- `max_gap 50` fills more frames but some of those straight lines are invented (ball in the air or out of view).

## 7. Render the video and save the tables

In [ ]:
team_ann = TeamAnnotator(TEAM_COLOURS, TEAM_NAMES)
poss_ann = PossessionAnnotator(TEAM_COLOURS, TEAM_NAMES)
pct = running_possession(poss)
people = df[df.tracker_id >= 0]
by_frame = {f: g for f, g in people.groupby("frame")}
out_path = OUT / "m5_possession.mp4"
with VideoWriter(out_path, info["fps"], info["width"], info["height"]) as w:
    for i, frame in iter_frames(VIDEO):
        g = by_frame.get(i)
        img = team_ann(frame, g, class_col=CLS) if g is not None else frame
        hid = poss.holder_id_s.iat[i] if i < len(poss) else -1
        hbox = None
        if hid >= 0 and g is not None:
            r = g[g.tracker_id == hid]
            if len(r): hbox = r[["x1", "y1", "x2", "y2"]].iloc[0].to_numpy()
        row = bi.iloc[i] if i < len(bi) else None
        team = poss.holder_team_s.iat[i] if i < len(poss) else -1
        w.write(poss_ann(img, row, hbox, team, pct.iloc[i].to_dict() if i < len(pct) else {}))
bi.to_csv(OUT / "m5_ball.csv")
poss.to_csv(OUT / "m5_possession.csv")
events.to_csv(OUT / "m5_events.csv", index=False)
print("saved", out_path, "+ m5_ball.csv, m5_possession.csv, m5_events.csv")

## 8. Check yourself

**Questions** (answer on paper first):
1. Why does a rolling **median** catch false balls without flagging a fast shot?
2. Why do we fill only gaps of at most 25 frames, and only with a ball known on both sides?
3. Why is the distance divided by the player's box height?
4. What does `min_frames` prevent? What happens with `min_frames = 1`?
5. During a long pass nobody is near the ball. Who is in control, and why?
6. Give one situation where the pixel-based rule picks the wrong holder.
7. How would you check whether a "turnover" in the table is real?

**Do it yourself**
- Fill in: ball detected ____% → known after cleaning ____% | possession ____ / ____ | passes ____ | turnovers ____.
- Watch `m5_possession.mp4`. Write down 2 moments where the triangle (holder) is wrong and why.

**Pitfalls**
- Interpolating across a camera cut draws a line between two unrelated positions. Long gaps are skipped for this reason.
- High balls and headers: the ball is above a player but far from his feet in the air. The rule uses feet, so these are often missed; that is acceptable for possession.
- Possession % depends on the settings. Report it with the settings and the stability table from section 6.